Generate raw object with raw integer counts for all genes and all metadata, subsetted on filtered cells in finalized object.

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import gc
import os

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)

### Read in raw object

In [3]:
adata = sc.read_h5ad('../data/02b_integration/01b_trimmed.h5ad')

In [4]:
adata.shape

(3017191, 25141)

### Read in metadata from finalized object

In [5]:
meta = pd.read_csv('../data/02b_integration/09_final_full-1/09_final_full-1-metadata.csv', index_col=0)

/tmp/ipykernel_28817/2198700860.py:1: DtypeWarning: Columns (27,28,32,33,36,37,46,50,52,54,55,56,63) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv('../data/02b_integration/09_final_full-1/09_final_full-1-metadata.csv', index_col=0)


### Subset out low quality cells in raw object

In [6]:
adata = adata[adata.obs.index.isin(meta.index)].copy()

In [7]:
adata.shape

(2452841, 25141)

### Remove columns from adata that are in meta

In [8]:
columns_to_remove = meta.columns.intersection(adata.obs.columns)

In [9]:
adata.obs.drop(columns=columns_to_remove, inplace=True)

### Transfer metadata from finalized object to raw object

In [10]:
adata.obs = adata.obs.merge(meta, left_index=True, right_index=True, how='left')

In [11]:
adata

AnnData object with n_obs × n_vars = 2452841 × 25141
    obs: 'name', 'library_id', 'individual', 'protocol', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'pct_counts_ribo', 'n_counts', '_scvi_batch', '_scvi_labels', 'Level_1', 'Level_2', 'Level_3', 'Level_4', 'Level_5', 'Level_6', 'ICU_stay', 'ICU_day', 'Episode_id', 'bal_barcode', 'Binary_outcome', 'Discharge_disposition', 'Episode_etiology', 'Immunocompromised_flag', 'External_transfer_flag', 'SOFA_score', 'viral_pathogen', 'pathogen_groups', 'clear_cut', 'number_of_pathogens', 'episode_type', 'clinical_opinion_l2', 'day_of_hospitalization', 'week_of_hospitalization', 'days_on_ventilator', 'day_since_first_intubation', 'mean_nat_score_until_yesterday', 'episode_counter', 'future_episode_outcome', 'is_episode_cured', 'future_vap_onset', 'vap_onset', 'number_of_bals', 'number_of_episodes', 'total_pathogens', 'number_of_tota

In [12]:
adata.var.tail(5)

,gene_ids,feature_types,genome,mito,ribo,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells
AC023491.2,GRCh38______ENSG00000278633,Gene Expression,GRCh38,False,False,103,0.000035,99.996592,106.0,103
AC004556.1,GRCh38______ENSG00000276345,Gene Expression,GRCh38,False,False,27107,0.009447,99.103192,28556.0,27107
AC233755.2,GRCh38______ENSG00000277856,Gene Expression,GRCh38,False,False,4635,0.215264,99.846656,650660.0,4635
AC233755.1,GRCh38______ENSG00000275063,Gene Expression,GRCh38,False,False,11563,0.500670,99.617450,1513329.0,11563
AC240274.1,GRCh38______ENSG00000271254,Gene Expression,GRCh38,False,False,34768,0.011878,98.849735,35904.0,34768


### Remove "GRCh38______" prefix from gene names

In [13]:
adata.var.gene_ids = [i.split('GRCh38______')[1] if i.startswith('GRCh38______') else i for i in adata.var.gene_ids]

In [14]:
adata.var.tail(5)

,gene_ids,feature_types,genome,mito,ribo,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,n_cells
AC023491.2,ENSG00000278633,Gene Expression,GRCh38,False,False,103,0.000035,99.996592,106.0,103
AC004556.1,ENSG00000276345,Gene Expression,GRCh38,False,False,27107,0.009447,99.103192,28556.0,27107
AC233755.2,ENSG00000277856,Gene Expression,GRCh38,False,False,4635,0.215264,99.846656,650660.0,4635
AC233755.1,ENSG00000275063,Gene Expression,GRCh38,False,False,11563,0.500670,99.617450,1513329.0,11563
AC240274.1,ENSG00000271254,Gene Expression,GRCh38,False,False,34768,0.011878,98.849735,35904.0,34768


## Remove transcripts with 'AC', 'AP', 'AL', 'AF', etc. prefix from list of genes. Also remove ribosomal genes.
These are "genes" are actually transcripts with prefixes like ACXXXXX, APXXXXXX, and ALXXXXXX. See Biostars note [here](https://www.biostars.org/p/9553891/). Since these are not biologically significant genes, it makes sense to exclude them. However, we don't expect to find them expressed since they are uniquely detected in 3' chemistry.

Ribosomal genes are being filtered here as they may need to be removed for model specific preprocessing for downstream analysis.

In [16]:
# specify the prefix pattern
prefixes_to_remove = ['AC', 'AL', 'AP', 'AF', 'AD', 'BX', 'CR', 'FP', 'KF']

# specify tlhe pattern to match genes to remove
pattern_to_remove = '|'.join([f'{prefix}\d{{6}}\.\d' for prefix in prefixes_to_remove] + ['Z\d{5}\.\d', 'U\d{5}\.\d'])

In [17]:
# Create boolean masks for both conditions
pattern_mask = adata.var.index.str.contains(pattern_to_remove, regex=True)
ribo_mask = adata.var.index.str.startswith(("RPS", "RPL"))

# Create a combined mask using numpy.logical_or
combined_mask = np.logical_or(pattern_mask, ribo_mask)

In [18]:
ribo_mask

array([False, False, False, ..., False, False, False])

In [19]:
adata

AnnData object with n_obs × n_vars = 2452841 × 25141
    obs: 'name', 'library_id', 'individual', 'protocol', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'pct_counts_ribo', 'n_counts', '_scvi_batch', '_scvi_labels', 'Level_1', 'Level_2', 'Level_3', 'Level_4', 'Level_5', 'Level_6', 'ICU_stay', 'ICU_day', 'Episode_id', 'bal_barcode', 'Binary_outcome', 'Discharge_disposition', 'Episode_etiology', 'Immunocompromised_flag', 'External_transfer_flag', 'SOFA_score', 'viral_pathogen', 'pathogen_groups', 'clear_cut', 'number_of_pathogens', 'episode_type', 'clinical_opinion_l2', 'day_of_hospitalization', 'week_of_hospitalization', 'days_on_ventilator', 'day_since_first_intubation', 'mean_nat_score_until_yesterday', 'episode_counter', 'future_episode_outcome', 'is_episode_cured', 'future_vap_onset', 'vap_onset', 'number_of_bals', 'number_of_episodes', 'total_pathogens', 'number_of_tota

In [20]:
adata = adata[:, ~combined_mask].copy()

In [21]:
adata.shape

(2452841, 19488)

### Save object

In [22]:
adata.obs['patient'] = adata.obs['patient'].astype(str)

In [23]:
adata.write('../data/02b_integration/09_raw/09_raw.h5ad')